# Project 06: End-to-End Enterprise Employee Attrition Prediction System
### AI/ML Engineer Portfolio – Supervised Machine Learning & People Analytics

**Author:** Senior Machine Learning Engineer  
**Category:** 03 - Machine Learning  
**Level:** Intermediate to Advanced  

---

## 1. Problem Statement & Business Context
Unplanned employee turnover costs modern enterprises between 1.5x to 2x an employee's annual salary when factoring in recruiting, onboarding, lost domain knowledge, and productivity lags.

This project builds an enterprise-grade, end-to-end Machine Learning pipeline to:
1. Quantify individual employee flight risk before voluntary resignation occurs.
2. Diagnose the principal organizational drivers influencing turnover (overtime, compensation, career stagnation, work-life balance).
3. Enforce ethical AI safeguards and conduct demographic fairness audits (EEOC 80% disparate impact standard).
4. Formulate proactive, empathetic retention intervention playbooks for HR leaders.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import load_attrition_data, generate_attrition_dataset
from src.data_preprocessing import clean_attrition_data, prepare_attrition_train_test
from src.model_training import train_baseline_models, tune_best_model_grid_search, load_attrition_pipeline
from src.model_evaluation import evaluate_attrition_models, extract_top_feature_importances, conduct_demographic_fairness_audit
from src.predictor import AttritionPredictor

print("Modules successfully imported.")

## 2. Data Acquisition & Exploratory Data Analysis (EDA)
We load employee records modeled after the industry-standard IBM HR Analytics benchmark, comprising 23 professional and demographic attributes.

In [ ]:
df_raw = load_attrition_data()
print(f"Raw dataset dimensions: {df_raw.shape}")
df_clean, audit_rep = clean_attrition_data(df_raw)
print(f"Cleaned dataset dimensions: {df_clean.shape}")
print(f"Initial records: {audit_rep['initial_records']}, Duplicates dropped: {audit_rep['duplicates_removed']}")
print(f"Overall workforce attrition prevalence: {audit_rep['attrition_rate_pct']}%")
df_clean.head()

## 3. Stratified Train-Test Splitting & Leak-Free Pipeline
Because voluntary attrition represents ~16% of the workforce, we use stratified splitting to guarantee equivalent class proportions across training and held-out test partitions.

In [ ]:
X_train, X_test, y_train, y_test = prepare_attrition_train_test(df_clean, test_size=0.20, random_state=42)
print(f"Training set: {X_train.shape[0]} records, Attrition rate: {y_train.mean():.2%}")
print(f"Testing set:  {X_test.shape[0]} records, Attrition rate: {y_test.mean():.2%}")

## 4. 5-Fold Stratified Cross-Validation Benchmark
We train three candidate classification architectures:
- Logistic Regression (L2 regularized, balanced)
- Decision Tree Classifier (balanced)
- Random Forest Classifier (balanced ensemble)

In [ ]:
candidate_pipes, cv_results = train_baseline_models(X_train, y_train, cv_folds=5, random_state=42)
print("5-Fold Cross-Validation Performance Ranking:")
cv_results

## 5. Hyperparameter Tuning via GridSearchCV
We optimize tree depth, estimator count, and minimum leaf samples to maximize discrimination capability while bounding variance.

In [ ]:
best_pipe, best_params, best_cv_score = tune_best_model_grid_search(X_train, y_train, cv_folds=5, random_state=42)
print(f"Best Cross-Validation ROC-AUC: {best_cv_score:.4f}")
print(f"Optimal Hyperparameters: {best_params}")

## 6. Held-Out Evaluation & Multi-Model Benchmark
We evaluate all models on the unseen 20% test partition across Accuracy, Precision, Recall, F1-Score, ROC-AUC, and PR-AUC.

In [ ]:
eval_dict = candidate_pipes.copy()
eval_dict["Tuned Random Forest"] = best_pipe
benchmark_df, preds, probs = evaluate_attrition_models(eval_dict, X_test, y_test)
benchmark_df

## 7. Feature Importance & Turnover Drivers
Extracting Gini importance values from the tuned Random Forest reveals the primary drivers of voluntary employee turnover.

In [ ]:
top_feats = extract_top_feature_importances(best_pipe, top_n=12)
print("Top 12 Turnover Drivers:")
top_feats

## 8. Responsible AI & Demographic Fairness Audit
We conduct a disparate impact audit across Gender pursuant to the EEOC Four-Fifths (80%) Rule to verify that the model does not disproportionately flag any protected group.

In [ ]:
fairness_df = conduct_demographic_fairness_audit(best_pipe, X_test, sensitive_col="Gender")
print("Demographic Disparate Impact Audit:")
fairness_df

## 9. Production Inference & Proactive Retention Playbooks
We instantiate `AttritionPredictor` to demonstrate real-time scoring and automated action planning for at-risk employees.

In [ ]:
predictor = AttritionPredictor()

at_risk_employee = {
    "Age": 29,
    "Gender": "Female",
    "MaritalStatus": "Single",
    "Department": "Sales",
    "JobRole": "Sales Representative",
    "JobLevel": 1,
    "MonthlyIncome": 2850,
    "TotalWorkingYears": 4,
    "YearsAtCompany": 2,
    "YearsInCurrentRole": 2,
    "YearsSinceLastPromotion": 5,
    "YearsWithCurrManager": 1,
    "DistanceFromHome": 26,
    "NumCompaniesWorked": 3,
    "JobSatisfaction": 1,
    "EnvironmentSatisfaction": 2,
    "WorkLifeBalance": 1,
    "JobInvolvement": 2,
    "OverTime": "Yes",
    "BusinessTravel": "Travel_Frequently",
    "StockOptionLevel": 0,
}

report = predictor.score_employee(at_risk_employee)
print(f"Attrition Probability: {report['attrition_probability_pct']}")
print(f"Risk Tier:             {report['risk_tier']}")
print(f"Recommended Action:    {report['recommended_action']}")
print("\nDetected Flight Factors:")
for f in report['risk_factors']:
    print(f" - {f}")